# QueueLess — EDA and waiting-time model walkthrough

> **Simulated data only.** This notebook is not executed in the repository; run it top to bottom after `pip install -r backend/requirements.txt jupyter`. Start Jupyter from the repo root so the relative paths resolve.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('backend').resolve()))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from app.ml.simulation import simulate, SimConfig
from app.ml.features import WAIT_FEATURES, WAIT_TARGET


## 1. Simulate
Discrete-event simulation; wait labels are *measured* from the event process, not computed from the features.

In [ ]:
sim = simulate(SimConfig(days=140, seed=42))
df = sim.tokens
print(len(df), 'tokens')
df.outcome.value_counts(normalize=True)

## 2. Target distribution (served tokens only — cancelled/no-show are censored)

In [ ]:
served = df[df.outcome=='served']
print(served[WAIT_TARGET].describe())
served[WAIT_TARGET].hist(bins=60); plt.xlabel('wait (min)'); plt.show()

## 3. Wait by hour and weekday

In [ ]:
served.groupby('hour')[WAIT_TARGET].mean().plot(kind='bar', title='Mean wait by hour'); plt.show()
served.groupby('day_of_week')[WAIT_TARGET].mean().plot(kind='bar', title='Mean wait by weekday (0=Mon)'); plt.show()

## 4. Feature relationships

In [ ]:
print(served[WAIT_FEATURES + [WAIT_TARGET]].corr(numeric_only=True)[WAIT_TARGET].sort_values(ascending=False))
served.plot.scatter('customers_ahead', WAIT_TARGET, s=3, alpha=.2); plt.show()

## 5. Data quality checks

In [ ]:
print('duplicates:', served.token_id.duplicated().sum())
print('missing:', served[WAIT_FEATURES + [WAIT_TARGET]].isna().sum().sum())
print('negative waits:', (served[WAIT_TARGET] < 0).sum())
print('wait > 4h:', (served[WAIT_TARGET] > 240).sum())

## 6. Train and compare models
The full benchmark (LR, RF, GBR, XGBoost, plus the non-ML fallback baseline) lives in `ml/train_wait_time.py` so results are reproducible outside the notebook.

In [ ]:
import subprocess; print(subprocess.run([sys.executable, 'ml/train_wait_time.py'], capture_output=True, text=True).stdout[-2500:])

In [ ]:
from IPython.display import Markdown
Markdown(open('ml/reports/wait_time_report.md', encoding='utf-8').read())